In [ ]:
# =====================================================================
# ÉTAPE 1 : Importation des bibliothèques
# =====================================================================
import pandas as pd
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt

# Style graphique pour les tracés
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# =====================================================================
# ÉTAPE 2 : Nettoyage, préparation des données et calcul de l'effort
# =====================================================================
print("Chargement et préparation des données...")

# Chargement des fichiers
wolf_df = pd.read_csv('DenaliWolfPopulation.csv')
bird_df = pd.read_csv('sample_occurrence.csv', low_memory=False)

# Si le nombre d'individus n'est pas précisé, on suppose qu'il y a au moins 1 oiseau
bird_df['individualCount'] = bird_df['individualCount'].fillna(1)

# A. Calcul du nombre total d'oiseaux par an
bird_counts = bird_df.groupby('year')['individualCount'].sum().reset_index(name='Bird_Count')

# B. Calcul de l'effort d'observation (Nombre d'observateurs uniques par an)
effort_df = bird_df.groupby('year')['recordedBy'].nunique().reset_index(name='Unique_Observers')

# C. Fusion des données ornithologiques (Comptages + Effort)
birds_summary = pd.merge(bird_counts, effort_df, on='year')

# D. Fusion finale avec les données des loups
df = pd.merge(wolf_df, birds_summary, left_on='Year', right_on='year', how='inner')

# E. Standardisation (Z-score) des variables explicatives pour aider la convergence du modèle MCMC
df['Wolves_Scaled'] = (df['Wolves_Counted_Fall'] - df['Wolves_Counted_Fall'].mean()) / df['Wolves_Counted_Fall'].std()
df['Effort_Scaled'] = (df['Unique_Observers'] - df['Unique_Observers'].mean()) / df['Unique_Observers'].std()

# Extraction sous forme de tableaux NumPy (Requis par PyMC)
wolves_x = df['Wolves_Scaled'].values
effort_x = df['Effort_Scaled'].values
birds_y = df['Bird_Count'].values

In [ ]:
# =====================================================================
# ÉTAPE 3 : Définition et échantillonnage du Modèle Bayésien
# =====================================================================
print("\nLancement de l'échantillonnage MCMC...")

with pm.Model() as corrected_model:
    # 1. Priors (A priori)
    # L'intercept de base
    alpha = pm.Normal('alpha', mu=np.log(birds_y.mean()), sigma=2)
    
    # L'effet causal de la population de loups
    beta_wolves = pm.Normal('beta_wolves', mu=0, sigma=1) 
    
    # L'effet du biais d'observation (le nombre d'observateurs)
    beta_effort = pm.Normal('beta_effort', mu=0, sigma=1)
    
    # Paramètre de surdispersion de la loi Binomiale Négative
    alpha_nb = pm.Exponential('alpha_nb', 1)
    
    # 2. Fonction de lien exponentielle (log-link) combinant les deux variables
    mu = pm.math.exp(alpha + beta_wolves * wolves_x + beta_effort * effort_x)

    # 3. Vraisemblance (Likelihood) basée sur la distribution Binomiale Négative
    y_obs = pm.NegativeBinomial('y_obs', mu=mu, alpha=alpha_nb, observed=birds_y)
    
    # 4. Échantillonnage
    trace = pm.sample(draws=2000, tune=1000, chains=4, target_accept=0.9, return_inferencedata=True)
    
    # 5. Génération des prédictions du modèle (pour le Posterior Predictive Check)
    ppc = pm.sample_posterior_predictive(trace)

In [ ]:
# =====================================================================
# ÉTAPE 4 : Analyses et Graphiques de diagnostic
# =====================================================================
print("\nGénération des analyses et graphiques...")

# A. Résumé statistique des paramètres
print("\nRésumé des paramètres estimés :")
display(az.summary(trace, var_names=['beta_wolves', 'beta_effort']))

# B. Forest Plot : Comparaison de l'effet des Loups vs l'Effort
# (Correction appliquée : retrait de hdi_prob pour la compatibilité ArviZ)
az.plot_forest(trace, var_names=['beta_wolves', 'beta_effort'], combined=True)
plt.axvline(0, color='red', linestyle='--', label='Zéro (Aucun effet)')
plt.title("Distribution a posteriori : Loups vs Effort d'observation")
plt.legend()
plt.show()

# C. Posterior Predictive Check (Version robuste en pur Matplotlib)
# Extraction des échantillons prédictifs
ppc_samples = ppc.posterior_predictive['y_obs'].values
ppc_samples_flat = ppc_samples.reshape(-1, ppc_samples.shape[-1])

# Sélection aléatoire de 100 simulations
rng = np.random.default_rng(42)
idx = rng.choice(ppc_samples_flat.shape[0], size=100, replace=False)

plt.figure(figsize=(10, 5))

# Tracé des simulations (lignes fines bleues)
for i in idx:
    counts, bins = np.histogram(ppc_samples_flat[i], bins=20, range=(0, birds_y.max() * 1.2))
    plt.plot(bins[:-1], counts, color='lightblue', alpha=0.3, drawstyle='steps-post')

In [ ]:
# Tracé des données réelles (ligne noire épaisse)
counts_obs, bins_obs = np.histogram(birds_y, bins=20, range=(0, birds_y.max() * 1.2))
plt.plot(bins_obs[:-1], counts_obs, color='black', lw=2.5, drawstyle='steps-post', label='Données réelles')

# Ligne factice pour la légende des simulations
plt.plot([], [], color='lightblue', alpha=0.7, label='Simulations du modèle (PPC)')

plt.title("Posterior Predictive Check (PPC) : Modèle corrigé")
plt.xlabel("Nombre d'oiseaux observés (par an)")
plt.ylabel("Fréquence")
plt.legend()
plt.show()